# Results Analysis & Thesis Figures
### Master's Research: Machine Unlearning — Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

This notebook is **read-only with respect to training** — it only loads saved checkpoints and produces plots.  
All heavy computation is done by the CLI scripts via `runner_kaggle.ipynb`:
- `train.py` — standard training
- `unlearn_naive.py` — naive machine unlearning
- `train_sisa.py` — SISA ensemble training
- `unlearn_sisa.py` — SISA-based machine unlearning

> **Before running:** make sure you have run `runner_kaggle.ipynb` at least once  
> so that checkpoints exist in the checkpoint directory.


## 0. Configuration — set paths here
Paths are pre-configured for Kaggle. Uncomment the relevant block below if running locally or on Colab.


In [ ]:
import os, sys

# ── Kaggle paths ────────────────────────────────────────────────────
# CKPT_DIR  = '/kaggle/working/checkpoints'
# DATA_ROOT = '/kaggle/working/data'

# ── Local paths (uncomment when running locally) ────────────────────
CKPT_DIR  = '../checkpoints'
DATA_ROOT = '../data'

# ── Colab + Google Drive (uncomment when running on Colab) ──────────
# CKPT_DIR  = '/content/drive/MyDrive/master_thesis/checkpoints'
# DATA_ROOT = '/content/drive/MyDrive/master_thesis/data'

DATASET     = 'CIFAR10'       # 'CIFAR10' | 'CIFAR100'
NUM_CLASSES = 10 if DATASET == 'CIFAR10' else 100
ds_tag      = DATASET.lower()

# SISA results live in a sub-directory per dataset
SISA_DIR = os.path.join(CKPT_DIR, f'sisa_{ds_tag}')

# Add repo root to path (needed on Kaggle / Colab)
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(''), '..')))

print(f'Checkpoint dir : {os.path.abspath(CKPT_DIR)}')
print(f'SISA dir       : {os.path.abspath(SISA_DIR)}')
print(f'Dataset        : {DATASET}')


## 1. Imports

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from utils import (STATS, evaluate, get_datasets, get_test_transform,
                   load_checkpoint, per_class_accuracy)

plt.rcParams.update({
    'figure.dpi':     120,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'font.size':          11,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 2. Load Checkpoints

In [ ]:
original_ckpt      = os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_best.pth')
naive_unlearn_ckpt = os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_naive_unlearn_best.pth')

# ── Original model ───────────────────────────────────────────────────
print('Loading original model...')
original_model  = load_checkpoint(original_ckpt, device)
orig_ckpt_data  = torch.load(original_ckpt, map_location='cpu')
orig_history    = orig_ckpt_data.get('history', {})
orig_time       = orig_ckpt_data.get('train_time_s', 0)

# ── Naive unlearned model ────────────────────────────────────────────
naive_model   = None
naive_history = {}
naive_cfg     = {}
naive_time    = 0

if os.path.exists(naive_unlearn_ckpt):
    print('\nLoading naive unlearned model...')
    naive_model     = load_checkpoint(naive_unlearn_ckpt, device)
    naive_ckpt_data = torch.load(naive_unlearn_ckpt, map_location='cpu')
    naive_history   = naive_ckpt_data.get('history', {})
    naive_cfg       = naive_ckpt_data.get('config',  {})
    naive_time      = naive_ckpt_data.get('unlearn_time_s', 0)

    forget_strategy = naive_cfg.get('forget_strategy', 'unknown')
    forget_size     = naive_ckpt_data.get('forget_size', '?')
    retain_size     = naive_ckpt_data.get('retain_size', '?')
    print(f'\nForget strategy : {forget_strategy}')
    print(f'Forget set size : {forget_size}')
    print(f'Retain set size : {retain_size}')
else:
    print('\nNo naive unlearned model found. Only evaluating original model.')
    naive_cfg = {'seed': 42, 'forget_strategy': 'random',
                 'forget_fraction': 0.01, 'batch_size': 256}
    forget_strategy = 'random'
    forget_size, retain_size = '?', '?'

# ── SISA results ─────────────────────────────────────────────────────
sisa_results      = {}
sisa_results_path = os.path.join(SISA_DIR, 'unlearn_results.json')

if os.path.exists(sisa_results_path):
    print(f'\nLoading SISA results from {sisa_results_path}...')
    with open(sisa_results_path, 'r') as f:
        sisa_results = json.load(f)
    print(f'SISA strategy     : {sisa_results.get("forget_strategy")}')
    print(f'SISA shards/slices: {sisa_results.get("num_shards")}/{sisa_results.get("num_slices")}')
    print(f'SISA unlearn time : {sisa_results.get("unlearn_time_s", 0):.2f}s')
else:
    print('\nNo SISA results found. Run unlearn_sisa.py first.')


## 3. Build Data Splits

In [ ]:
import random, torchvision

full_train, test_ds = get_datasets(DATASET, DATA_ROOT)
DatasetClass = (torchvision.datasets.CIFAR10 if DATASET == 'CIFAR10'
                else torchvision.datasets.CIFAR100)
full_eval = DatasetClass(root=DATA_ROOT, train=True, download=True,
                         transform=get_test_transform(DATASET))

# Reconstruct forget / retain split using same seed as unlearning script
seed = naive_cfg.get('seed', 42)
rng  = random.Random(seed)
all_indices = list(range(len(full_train)))

if forget_strategy == 'random':
    frac = naive_cfg.get('forget_fraction', 0.01)
    forget_indices = rng.sample(all_indices, int(len(all_indices) * frac))
elif forget_strategy == 'class':
    cls = naive_cfg.get('forget_class', 0)
    forget_indices = [i for i, (_, label) in enumerate(full_train) if label == cls]
else:
    forget_indices = []

forget_set     = set(forget_indices)
retain_indices = [i for i in all_indices if i not in forget_set]

bs = naive_cfg.get('batch_size', 256)
retain_eval_loader = DataLoader(Subset(full_eval, retain_indices),
                                batch_size=bs, shuffle=False, num_workers=2)
forget_loader      = DataLoader(Subset(full_eval, forget_indices),
                                batch_size=bs, shuffle=False, num_workers=2)
test_loader        = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2)

print(f'Forget : {len(forget_indices):,}  |  Retain : {len(retain_indices):,}  |  Test : {len(test_ds):,}')


## 4. Evaluation — All Splits

In [ ]:
criterion = nn.CrossEntropyLoss()

# ── Original ─────────────────────────────────────────────────────────
orig_retain_loss, orig_retain_acc = evaluate(original_model, retain_eval_loader, criterion, device)
orig_forget_loss, orig_forget_acc = evaluate(original_model, forget_loader,      criterion, device)
orig_test_loss,   orig_test_acc   = evaluate(original_model, test_loader,        criterion, device)

print(f"{'='*62}")
print(f"{'Metric':<20} {'Original':>12}")
print(f"{'='*62}")
print(f"{'Retain Accuracy':<20} {orig_retain_acc:>11.2f}%")
print(f"{'Forget Accuracy':<20} {orig_forget_acc:>11.2f}%")
print(f"{'Test Accuracy':<20} {orig_test_acc:>11.2f}%")

# ── Naive + SISA comparison ───────────────────────────────────────────
if naive_model is not None:
    naive_retain_loss, naive_retain_acc = evaluate(naive_model, retain_eval_loader, criterion, device)
    naive_forget_loss, naive_forget_acc = evaluate(naive_model, forget_loader,      criterion, device)
    naive_test_loss,   naive_test_acc   = evaluate(naive_model, test_loader,        criterion, device)

    sisa_after      = sisa_results.get('after', {})
    sisa_retain_acc = sisa_after.get('retain_acc', float('nan'))
    sisa_forget_acc = sisa_after.get('forget_acc', float('nan'))
    sisa_test_acc   = sisa_after.get('test_acc',   float('nan'))
    sisa_time       = sisa_results.get('unlearn_time_s', float('nan'))
    sisa_train_time = sisa_results.get('sisa_train_time_s', 0)
    have_sisa       = bool(sisa_results)

    if have_sisa:
        # ── Accuracy table ────────────────────────────────────────────
        print(f"\n{'='*92}")
        print(f"{'Metric':<20} {'Original':>12}  {'Naive Retrain':>14}  {'SISA Unlearn':>14}  {'Δ Naive':>8}  {'Δ SISA':>8}")
        print(f"{'='*92}")
        rows = [
            ('Retain Accuracy', orig_retain_acc, naive_retain_acc, sisa_retain_acc),
            ('Forget Accuracy', orig_forget_acc, naive_forget_acc, sisa_forget_acc),
            ('Test Accuracy',   orig_test_acc,   naive_test_acc,   sisa_test_acc),
        ]
        for name, orig, naive, sisa in rows:
            d_naive = naive - orig
            d_sisa  = sisa  - orig
            s_naive = '+' if d_naive >= 0 else ''
            s_sisa  = '+' if d_sisa  >= 0 else ''
            print(f'{name:<20} {orig:>11.2f}%  {naive:>13.2f}%  {sisa:>13.2f}%  {s_naive}{d_naive:>7.2f}%  {s_sisa}{d_sisa:>7.2f}%')
        print(f"{'='*92}")

        # ── Timing table ─────────────────────────────────────────────
        print(f"\n{'='*70}")
        print(f"{'Method':<30} {'Training Time':>18} {'Unlearn Time':>18}")
        print(f"{'='*70}")
        print(f"{'Original / Naive Retrain':<30} {orig_time:>17.1f}s {naive_time:>17.1f}s")
        print(f"{'SISA Ensemble':<30} {sisa_train_time:>17.1f}s {sisa_time:>17.1f}s")
        print(f"{'='*70}")
        if naive_time > 0 and sisa_time > 0:
            speedup = naive_time / sisa_time
            print(f"Unlearning speedup (SISA vs Naive): {speedup:.1f}x")
    else:
        # ── Accuracy table (no SISA) ──────────────────────────────────
        print(f"\n{'='*62}")
        print(f"{'Metric':<20} {'Original':>12}  {'Naive Retrain':>14}  {'Δ Naive':>8}")
        print(f"{'='*62}")
        rows = [
            ('Retain Accuracy', orig_retain_acc, naive_retain_acc),
            ('Forget Accuracy', orig_forget_acc, naive_forget_acc),
            ('Test Accuracy',   orig_test_acc,   naive_test_acc),
        ]
        for name, orig, naive in rows:
            d_naive = naive - orig
            s_naive = '+' if d_naive >= 0 else ''
            print(f'{name:<20} {orig:>11.2f}%  {naive:>13.2f}%  {s_naive}{d_naive:>7.2f}%')
        print(f"{'='*62}")


## 5. Training Curves

In [ ]:
def plot_curves(history, title, best_acc=None):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train', linewidth=2)
    axes[0].plot(epochs, history['test_loss'],  label='Test',  linewidth=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title(f'{title} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    best_label = f'Test (best: {best_acc:.2f}%)' if best_acc else 'Test'
    axes[1].plot(epochs, history['train_acc'], label='Train',      linewidth=2)
    axes[1].plot(epochs, history['test_acc'],  label=best_label,   linewidth=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title(f'{title} — Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    return fig

if orig_history:
    fig = plot_curves(orig_history, f'{DATASET} — Original Training',
                      best_acc=orig_ckpt_data.get('test_acc'))
    fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_training_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history found in original checkpoint.')


In [ ]:
if naive_history:
    fig = plot_curves(naive_history,
                      f'{DATASET} — Naive Unlearning ({forget_strategy}, {forget_size} samples)',
                      best_acc=naive_ckpt_data.get('test_acc'))
    fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_naive_unlearn_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history found in naive checkpoint.')


## 6. Per-Class Accuracy

In [ ]:
orig_per_class  = per_class_accuracy(original_model, test_loader, NUM_CLASSES, device)

fig, ax = plt.subplots(figsize=(12, 4))
x = range(NUM_CLASSES)
ax.bar([i - 0.2 for i in x], orig_per_class, width=0.4, label='Original', alpha=0.8)

if naive_model is not None:
    naive_per_class = per_class_accuracy(naive_model, test_loader, NUM_CLASSES, device)
    ax.bar([i + 0.2 for i in x], naive_per_class, width=0.4, label='Naive Unlearn', alpha=0.8)

ax.set_xlabel('Class'); ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{DATASET} — Per-Class Test Accuracy')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_per_class_acc.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 7. SISA Specifics
Detailed view of the SISA ensemble structure and unlearning performance.
Only populated when `unlearn_sisa.py` has been run and its results loaded in Section 2.


In [ ]:
if sisa_results:
    print(f"Dataset          : {sisa_results['dataset']}")
    print(f"Shards / Slices  : {sisa_results['num_shards']} / {sisa_results['num_slices']}")
    print(f"Aggregation      : {sisa_results['aggregation']}")
    print(f"Affected Shards  : {sisa_results['affected_shards']}")
    print(f"Unlearning Time  : {sisa_results['unlearn_time_s']:.2f}s")
    
    before = sisa_results.get('before', {})
    after  = sisa_results.get('after',  {})
    print(f"\n{'='*60}")
    print(f"{'Metric':<22} {'Before':>12}  {'After':>12}  {'Δ':>8}")
    print(f"{'='*60}")
    for key, label in [('retain_acc', 'Retain Acc'), ('forget_acc', 'Forget Acc'), ('test_acc', 'Test Acc')]:
        b = before.get(key, float('nan'))
        a = after.get(key, float('nan'))
        d = a - b
        s = '+' if d >= 0 else ''
        print(f"{label:<22} {b:>11.2f}%  {a:>11.2f}%  {s}{d:>7.2f}%")
    print(f"{'='*60}")
    
    print("\nShard Stats:")
    print(f"{'Shard':<8} {'Affected':>10} {'Slices Retrained':>18} {'Time (s)':>10}")
    print("-" * 50)
    for shard in sisa_results.get('shard_stats', []):
        affected = "Yes" if shard['retrained'] else "No"
        slices_r = shard.get('slices_retrained', 0)
        t        = shard.get('retrain_time_s', 0)
        print(f"{shard['shard_id']:<8} {affected:>10} {slices_r:>18} {t:>10.1f}")
else:
    print("SISA results not available. Run 5g–5i in runner_kaggle.ipynb first.")


## 8. Bar Chart — Retain / Forget / Test Accuracy Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

methods  = ['Original']
retain_v = [orig_retain_acc]
forget_v = [orig_forget_acc]
test_v   = [orig_test_acc]

if naive_model is not None:
    methods.append('Naive Retrain')
    retain_v.append(naive_retain_acc)
    forget_v.append(naive_forget_acc)
    test_v.append(naive_test_acc)

if sisa_results:
    after = sisa_results.get('after', {})
    methods.append('SISA Unlearn')
    retain_v.append(after.get('retain_acc', 0))
    forget_v.append(after.get('forget_acc', 0))
    test_v.append(after.get('test_acc', 0))

x     = np.arange(len(methods))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, retain_v, width, label='Retain Acc', alpha=0.85)
ax.bar(x,         forget_v, width, label='Forget Acc', alpha=0.85)
ax.bar(x + width, test_v,   width, label='Test Acc',   alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{DATASET} — Unlearning Method Comparison')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
